In [1]:
# Install AutoGen Studio and pyngrok
!pip install -q autogenstudio pyngrok

# Verify the installation
!autogenstudio version

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.4/83.4 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.1/106.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.6/96.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.9/296.9 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
import os
import getpass

# 1. Set up Groq API Key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

# 2. Set up Ngrok Auth Token (Required to tunnel the AutoGen Studio UI)
NGROK_TOKEN = getpass.getpass("Enter your Ngrok Auth Token: ")
!ngrok config add-authtoken {NGROK_TOKEN}

Enter your Groq API Key: ··········
Enter your Ngrok Auth Token: ··········
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [3]:
import multiprocessing
import time
from pyngrok import ngrok

def run_autogen_studio():
    # Launch AutoGen Studio on port 8081
    !autogenstudio ui --port 8081 --host 0.0.0.0

# Start AutoGen Studio in a background process
process = multiprocessing.Process(target=run_autogen_studio)
process.start()

# Wait a moment for the server to spin up
time.sleep(5)

# Open the ngrok tunnel to port 8081
public_url = ngrok.connect(8081)
print("\n" + "="*60)
print(f"[SUCCESS] AutoGen Studio is running!")
print(f"Click the link below to open the UI:")
print(f"{public_url}")
print("="*60 + "\n")


[SUCCESS] AutoGen Studio is running!
Click the link below to open the UI:
NgrokTunnel: "https://persecute-treat-aqueduct.ngrok-free.dev" -> "http://localhost:8081"



In [6]:
import os
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient

async def run_team():
    # Fetch the Groq API key you securely entered in Cell 2
    groq_key = ""

    # Define Groq model clients with the correct model_info settings
    researcher_model = OpenAIChatCompletionClient(
        model="llama-3.1-8b-instant",
        base_url="https://api.groq.com/openai/v1",
        api_key=groq_key,
        model_info={
            "vision": False,
            "function_calling": True,
            "json_output": True,
            "structured_output": True,
            "family": "unknown"
        }
    )

    editor_model = OpenAIChatCompletionClient(
        model="llama-3.3-70b-versatile",
        base_url="https://api.groq.com/openai/v1",
        api_key=groq_key,
        model_info={
            "vision": False,
            "function_calling": True,
            "json_output": True,
            "structured_output": True,
            "family": "unknown"
        }
    )

    # Define the individual Agents
    researcher = AssistantAgent(
        name="Researcher",
        model_client=researcher_model,
        system_message="You are an expert researcher. Provide a highly detailed summary using clear Markdown formatting."
    )

    editor = AssistantAgent(
        name="Editor",
        model_client=editor_model,
        system_message="You are a strict editor. Critique the researcher's work and optimize it for professional delivery."
    )

    # Orchestrate the workflow team
    team = RoundRobinGroupChat(
        participants=[researcher, editor],
        termination_condition=MaxMessageTermination(max_messages=4)
    )

    # Run the prompt
    print("--- Starting Multi-Agent Session ---")
    async for message in team.run_stream(task="Explain why Groq LPUs provide higher throughput for LLMs than standard GPUs."):
        print(f"\n\033[1m[{message.source}]\033[0m: {message.content}")
        print("-" * 40)

# Execute the async loop inside Google Colab
await run_team()

--- Starting Multi-Agent Session ---

[user]: Explain why Groq LPUs provide higher throughput for LLMs than standard GPUs.
----------------------------------------

[Researcher]: **Groq LPUs and their Advantages over Standard GPUs for LLMs**

**Introduction**
---------------

Large Language Models (LLMs) have revolutionized the field of natural language processing, but they also pose significant computational challenges. Traditional Graphics Processing Units (GPUs) have been the primary choice for training and running LLMs due to their high processing capacities. However, a relatively new company called Groq has introduced Low-Power Units (LPUs), which claim to surpass traditional GPUs in terms of performance while being significantly more energy-efficient. In this summary, we will explore why Groq LPUs might provide higher throughput for LLMs than standard GPUs.

**Groq LPUs Architecture**
---------------------------

Groq's LPUs are custom-designed processors that combine aspects of 

AttributeError: 'TaskResult' object has no attribute 'source'

### Extension 1: Add a Tool-Wielding Agent

In [8]:
import asyncio
import os

from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_core.tools import FunctionTool
from autogen_ext.models.openai import OpenAIChatCompletionClient

# =====================================================
# Extension 1: Tool-Wielding Agent
# =====================================================

def stock_price_tool(company: str) -> str:
    """
    Sample stock lookup tool.
    """
    stock_prices = {
        "Apple": "$215.30",
        "Microsoft": "$498.10",
        "Google": "$183.75",
        "Amazon": "$226.40",
        "NVIDIA": "$171.25"
    }

    return stock_prices.get(
        company,
        f"No stock data available for {company}"
    )

stock_tool = FunctionTool(
    stock_price_tool,
    description="Returns the stock price of a company."
)

# Fetch the Groq API key from environment variables
groq_key = os.environ.get("GROQ_API_KEY")

# Define a generic model client for the agents
model_client = OpenAIChatCompletionClient(
    model="llama-3.1-8b-instant", # Using one of the Groq models
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_key,
    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": True,
        "structured_output": True,
        "family": "unknown"
    }
)

# =====================================================
# Researcher Agent (with Tool)
# =====================================================

researcher = AssistantAgent(
    name="Researcher",
    model_client=model_client,
    tools=[stock_tool],
    system_message="""
    You are a research specialist.

    If a user asks about stock prices,
    ALWAYS use stock_price_tool first.
    """
)

# =====================================================
# Editor Agent
# =====================================================

editor = AssistantAgent(
    name="Editor",
    model_client=model_client,
    system_message="""
    Improve and refine the research report.
    Make it professional and concise.
    """
)

In [10]:
# =====================================================
# Extension 2: User Proxy Agent
# =====================================================

user_proxy = UserProxyAgent(
    name="UserProxy",
    input_func=input
)

# =====================================================
# Team Configuration
# =====================================================

team = RoundRobinGroupChat(
    participants=[
        researcher,
        editor,
        user_proxy
    ],
    termination_condition=MaxMessageTermination(
        max_messages=8
    )
)

# =====================================================
# Run Workflow
# =====================================================

async def main():

    print("\n🚀 Starting Extended AutoGen + Groq Workflow\n")

    async for message in team.run_stream(
        task="""
        Find the stock price of Apple.
        Use the stock lookup tool.
        Create a short report.
        Let the editor improve it.
        Ask the user for approval before ending.
        """
    ):
        print(f"\n[{message.source}]")
        print(message.content)
        print("-" * 60)

await main()


🚀 Starting Extended AutoGen + Groq Workflow


[user]

        Find the stock price of Apple.
        Use the stock lookup tool.
        Create a short report.
        Let the editor improve it.
        Ask the user for approval before ending.
        
------------------------------------------------------------

[Researcher]
[FunctionCall(id='t9sk43yhv', arguments='{"company":"Apple"}', name='stock_price_tool')]
------------------------------------------------------------

[Researcher]
[FunctionExecutionResult(content='$215.30', name='stock_price_tool', call_id='t9sk43yhv', is_error=False)]
------------------------------------------------------------

[Researcher]
$215.30
------------------------------------------------------------

[Editor]
**Apple Stock Price Report**

**Company Name:** Apple Inc.
**Ticker Symbol:** AAPL
**Current Stock Price:** $215.30

**Market Status:** Current (as of the last available data)

This report provides a snapshot of Apple's current stock price, servin

AttributeError: 'TaskResult' object has no attribute 'source'